<!-- This is the beginning code to clean ebird for LA  -->

# **Part 1.** Cleaning Ebird Data 

Updated: 11.08.2026 
Author: Oliver Nguyen


* Warning to run this with more than 32 Gb of RAM otherwise it will overload the memory. 



This code borrows from eBird best practices and Callaghan et al. 2021 to work with citizen science data, we'll be using auk package to help us clean the data and using R. We downloaded respective datasets from San Francisco and Los Angeles. 


The main filters we will be using are: 

[x] Years 2010 - 2025 (Jan - Dec)  
[x] => 0.5 km distance  
[x] area <50 ha
[x] => 10 observers  
[x] combine shared checklists (Auk_Unique)  
[x] => 15- 120 minutes hours total efforts checklist  
[x] Stationary + Travelling + Area checklists  
[x] Completed only checklists  

We'll be aiming to be using **Incidence Frequency** instead of Abundance (Callaghan et al. 2022) as it would be more stable. 

According to Auk_ebd, the filters we can use are: 

#> Filters 
#>   Species: all
#>   Countries: USA
#>   States: California
#>   Counties: SF County/LA County
#>   BCRs: NA
#>   Bounding box: full extent
#>   Years: 2015 - 2026
#>   Date: all
#>   Start time: NA
#>   Last edited date: NA
#>   Protocol: Travelling, Stationary, Area
#>   Project code: NA
#>   Duration: 15 - 120 minutes
#>   Distance travelled: => 1 km 
#>   Records with breeding codes only: no
#>   Exotic Codes: all
#>   Complete checklists only: yes


In [2]:

#Libraries needed
import pandas as pd
import geopandas as gpd
from datetime import datetime
from pathlib import Path
import numpy as np
import duckdb as duckdb

# Import rpy2 components
#%conda install rpy2

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, Formula
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter


## Steps in Filtering


1. Import checklists and observations
2. Save to Parquets and process in chunks 
    * Convert and process effort hours 
    * auk roll-up 
    * zero-filling
4. Filter efforts (observation hours, distance, number of observers)
5. Seperate into stationary and travelling checklists
3. Keep only max observations of species per day 



In [3]:
#%conda install -c conda-forge r-arrow #If you need to install R packages, you ned to run conda 

In [4]:

# ============================================================================
# Import R packages
# ============================================================================

print("Loading R packages...")

# Import R packages
base = importr('base')
utils = importr('utils')
auk = importr('auk')
lubridate = importr('lubridate')
dplyr = importr('dplyr')
sf = importr('sf')
arrow = importr('arrow')

# Resolve namespace conflicts in R
ro.r('select <- dplyr::select')
ro.r('filter <- dplyr::filter')
print("R packages loaded successfully!")


Loading R packages...
R packages loaded successfully!


In [5]:

# ============================================================================
# eBird Data Processing with Auk Package
# ============================================================================

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from pathlib import Path
import pandas as pd



ro.r(r'''
    library(auk)
    library(dplyr)
    library(lubridate)
    library(sf)
    library(data.table)
    
    # Resolve namespace conflicts
    select <- dplyr::select
    filter <- dplyr::filter
    
    n_cores <- parallel::detectCores() - 1
    setDTthreads(n_cores)
    
    cat("Using", n_cores, "cores\n")
    

''')


R callback write-console: data.table 1.18.2.1 using 4 threads (see ?getDTthreads).    
R callback write-console: Latest news: r-datatable.com
  
R callback write-console: 
Attaching package: ‘data.table’

  
R callback write-console: The following objects are masked from ‘package:dplyr’:

    between, first, last

  
R callback write-console: The following objects are masked from ‘package:lubridate’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year

  


Using 7 cores


# Using Auk Package

Below are the approximate run times for each city: 

| City/Counties | Run Time (mins) |  Code    |
|-------|-----|-------|
| San Francisco | 8  | CA-075 |
| Los Angeles   |  28 | CA-037   |





In [ ]:
#Takes about 8 minutes 

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from pathlib import Path
import pandas as pd

ro.r(r'''
library(auk)
library(dplyr)
library(lubridate)
library(arrow)

#Import in the sampling and ebird data you got
#Reccomended to do this locally first, run the code for the city/uncomment as needed


#San Francisco (CA 075)
#f_ebd_input <- "/home/oliver/Documents/ebd_US-CA-075_201001_202612_smp_relSep-2026/ebd_US-CA-075_201001_202612_smp_relSep-2026.txt"
#f_sed_input <- "/home/oliver/Documents/ebd_US-CA-075_201001_202612_smp_relSep-2026/ebd_US-CA-075_201001_202612_smp_relSep-2026_sampling.txt"


#Los Angeles (CA 037)
f_ebd_input <- "/home/oliver/Documents/ebd_US-CA-037_201001_202612_smp_relSep-2026/ebd_US-CA-037_201001_202612_smp_relSep-2026.txt"
f_sed_input <- "/home/oliver/Documents/ebd_US-CA-037_201001_202612_smp_relSep-2026/ebd_US-CA-037_201001_202612_smp_relSep-2026_sampling.txt"

out_dir     <- "/home/oliver/Documents/data/output"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# New file names so the old stationary-only files aren't reused
f_ebd <- file.path(out_dir, "ebd_stat_trav_area.txt")
f_sed <- file.path(out_dir, "sed_stat_trav_area.txt")
zf_dir <- file.path(out_dir, "zf_stat_trav_area")

# ---- ONE pass through the raw files (filters auk can do) ----
if (!file.exists(f_ebd)) {
  auk_ebd(f_ebd_input, file_sampling = f_sed_input) |>
    auk_date(c("2010-01-01", "2025-12-31")) |>                  # matches the file and the loop years
    auk_protocol(c("Stationary", "Traveling", "Area")) |>        # eBird spelling: "Traveling"
    auk_duration(c(5, 120)) |>                                  # 5-120 minutes
    auk_complete() |>                                            # complete checklists only
    auk_filter(file = f_ebd, file_sampling = f_sed, overwrite = TRUE)
}

# ---- Read (unique = TRUE by default collapses shared/group checklists = auk_unique) ----
sed <- read_sampling(f_sed, unique = TRUE) |>
  filter(
    number_observers <= 10,
    case_when(
      protocol_name == "Stationary" ~ TRUE,
      protocol_name == "Traveling"  ~ !is.na(effort_distance_km) & effort_distance_km <= 0.5,
      protocol_name == "Area"       ~ !is.na(effort_area_ha)     & effort_area_ha < 50,
      TRUE ~ FALSE
    )
  )

ebd <- read_ebd(f_ebd, unique = TRUE) |>
  filter(checklist_id %in% sed$checklist_id)

# Check what survived each protocol
print(count(sed, protocol_name))
print(sed |> group_by(protocol_name) |>
        summarise(med_min  = median(duration_minutes),
                  med_km   = median(effort_distance_km, na.rm = TRUE),
                  med_ha   = median(effort_area_ha, na.rm = TRUE)))

time_to_decimal <- function(x) {
  x <- hms(x, quiet = TRUE); hour(x) + minute(x)/60 + second(x)/3600
}

# ---- Zero-fill per year, write Parquet ----
for (yr in 2010:2025) {
  sed_y <- filter(sed, year(observation_date) == yr)
  if (nrow(sed_y) == 0) next
  ebd_y <- filter(ebd, checklist_id %in% sed_y$checklist_id)

  zf <- auk_zerofill(ebd_y, sed_y, collapse = TRUE) |>
    mutate(
      observation_count  = if_else(species_observed,
                                   suppressWarnings(as.integer(observation_count)), 0L),  # "X" -> NA
      # keep real distances; stationary = 0, area = NA (area is in effort_area_ha)
      effort_distance_km = if_else(protocol_name == "Stationary", 0, effort_distance_km),
      effort_hours       = duration_minutes / 60,
      effort_speed_kmph  = effort_distance_km / effort_hours,
      hours_of_day       = time_to_decimal(time_observations_started),
      year               = year(observation_date),
      month              = month(observation_date),
      day_of_year        = yday(observation_date)
    )

  write_dataset(zf, zf_dir, format = "parquet",
                partitioning = c("year", "month"),
                existing_data_behavior = "delete_matching")
  cat(sprintf("%d: %s rows\n", yr, format(nrow(zf), big.mark = ",")))
  rm(zf, ebd_y, sed_y); gc()
}
''')



# A tibble: 3 × 2
  protocol_name      n
  <chr>          <int>
1 Area            3591
2 Stationary    126448
3 Traveling      74111
# A tibble: 3 × 4
  protocol_name med_min med_km med_ha
  <chr>           <dbl>  <dbl>  <dbl>
1 Area               54  0.685  2.02 
2 Stationary         20  0.169  0.809
3 Traveling          30  0.322  4    
2010: 966,069 rows
2011: 1,081,510 rows
2012: 1,550,230 rows
2013: 2,003,704 rows
2014: 2,500,608 rows
2015: 2,745,602 rows
2016: 3,236,688 rows
2017: 4,134,940 rows
2018: 4,868,336 rows
2019: 5,611,716 rows
2020: 7,365,728 rows
2021: 8,148,504 rows
2022: 8,015,322 rows
2023: 8,246,581 rows
2024: 9,512,235 rows
2025: 10,384,128 rows


R callback write-console: In addition:   
R callback write-console: There were 16 warnings (use warnings() to see them)  
R callback write-console: 
  


In [ ]:

import duckdb


zf_dir  = "/home/oliver/Documents/data/output/zf_stat_trav_area"

#SF output
#out_pq  = "/home/oliver/Documents/data/output/zf_SF_stat_trav_area_2010_2025.parquet" 

#LA output
out_pq  = "/home/oliver/Documents/data/output/zf_LA_stat_trav_area_2010_2025.parquet"

con = duckdb.connect()
con.execute(f"""
  COPY (
    SELECT *
    FROM read_parquet('{zf_dir}/**/*.parquet', hive_partitioning = true)
    ORDER BY observation_date
  ) TO '{out_pq}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Check: rows and checklists per year
print(con.execute(f"""
  SELECT year, COUNT(*) AS n_rows, COUNT(DISTINCT sampling_event_identifier) AS n_checklists
  FROM read_parquet('{out_pq}')
  GROUP BY year ORDER BY year
""").df())
con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

: 

Move to P2 from here. 

In [ ]:
#This part is to briefly look at our data sets
#free up some space 

ro.r(f'''
breaks <- seq(0, 24)
labels <- breaks[-length(breaks)] + diff(breaks) / 2
checklists_time <- checklists |> 
  mutate(hour_bins = cut(hours_of_day, 
                         breaks = breaks, 
                         labels = labels,
                         include.lowest = TRUE),
         hour_bins = as.numeric(as.character(hour_bins))) |> 
  group_by(hour_bins) |> 
  summarise(n_checklists = n(),
            n_detected = sum(species_observed),
            det_freq = mean(species_observed))
''')

In [ ]:

ro.r(r'''
# Get top 20 species by total observation count
top_species <- zf_filtered %>%
  group_by(scientific_name) %>%
  summarize(total_count = sum(observation_count), na.rm = TRUE) %>%
  arrange(desc(total_count)) %>%
  head(20) %>%
  pull(scientific_name)

top_species


# Plot only those top species
zf_filtered %>%
  filter(scientific_name %in% top_species) %>%
  ggplot(aes(x = observation_date, y = scientific_name)) +
  geom_tile() +
  scale_fill_viridis_c(name = "Year", option = "plasma") +
  labs(
    title = "Top 20 Bird Species Observations Over Time",
    x = "Date",
    y = "Species",
    fill = "Observation Count"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text.y = element_text(size = 10)
  )

#Get species sighted per day

top_species <- zf_filtered %>%
  group_by(observation_date) %>%
  summarize(total_count = count(unique(scientific_name)), na.rm = TRUE) %>%
  arrange(desc(total_count)) %>%
  head(20) %>%
  pull(scientific_name)

    '''
)



#Now let's look at Checklist variation 
ro.r(f'''
# summarize data by hour long bins
breaks <- seq(0, 6)
labels <- breaks[-length(breaks)] + diff(breaks) / 2
checklists_duration <- checklists |> 
  mutate(duration_bins = cut(effort_hours, 
                             breaks = breaks, 
                             labels = labels,
                             include.lowest = TRUE),
         duration_bins = as.numeric(as.character(duration_bins))) |> 
  group_by(duration_bins) |> 
  summarise(n_checklists = n(),
            n_detected = sum(species_observed),
            det_freq = mean(species_observed))

# histogram
g_duration_hist <- ggplot(checklists_duration) +
  aes(x = duration_bins, y = n_checklists) +
  geom_segment(aes(xend = duration_bins, y = 0, yend = n_checklists),
               color = "grey50") +
  geom_point() +
  scale_x_continuous(breaks = breaks) +
  scale_y_continuous(labels = scales::comma) +
  labs(x = "Checklist duration [hours]",
       y = "# checklists",
       title = "Distribution of checklist durations")

# frequency of detection
g_duration_freq <- ggplot(checklists_duration |> filter(n_checklists > 100)) +
  aes(x = duration_bins, y = det_freq) +
  geom_line() +
  geom_point() +
  scale_x_continuous(breaks = breaks) +
  scale_y_continuous(labels = scales::percent) +
  labs(x = "Checklist duration [hours]",
       y = "% checklists with detections",
       title = "Detection frequency")

# combine
grid.arrange(g_duration_hist, g_duration_freq)

''')

